# MLflow lesson: worked solutions to the exercises

Run this notebook top to bottom (about 3 minutes). It works in a **temporary MLflow
store**, so it changes nothing of yours: your `mlflow.db`, `mlruns/` and registered models are not touched.
Each exercise shows the question, the code that answers it, and an **Answer** based on the numbers printed above it.

In [1]:
import os, sys, shutil, subprocess, tempfile, warnings
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"
warnings.filterwarnings("ignore")

# A throwaway MLflow store (database + artifact folder) in a temp directory.
# The env var also covers the `train.py` subprocess in exercise 5.
TMP = Path(tempfile.mkdtemp(prefix="mlflow_solutions_"))
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{TMP / 'mlflow.db'}"

import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient

import emr_model as em
from train import EXPERIMENT, train_and_log

pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_colwidth", 200)
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
exp_id = mlflow.create_experiment(EXPERIMENT, artifact_location=(TMP / "artifacts").as_uri())
mlflow.set_experiment(EXPERIMENT)
client = MlflowClient()
print("temporary store:", TMP)

# Data: 2024 for notebook 1, Jan-Jun 2025 (concept_shift scenario) for notebook 2
em.simulate("2024-01", 12)
em.simulate("2025-01", 6, scenario="concept_shift")

2026/10/23 15:46:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/23 15:46:15 INFO mlflow.store.db.utils: Updating database tables


temporary store: /tmp/mlflow_solutions_5npjr_qn


## Setup: the runs from notebook 1

The exercises build on the runs from `1_tracking.ipynb`. Recreate them here: the baseline and the four-model grid
(same settings; `train_and_log` is the function the notebook wraps).

In [2]:
base_grid = [
    dict(model="logistic", C=1.0, run_name="baseline-logistic"),
    dict(model="logistic", C=0.1),
    dict(model="gbm", max_depth=2, learning_rate=0.05),
    dict(model="gbm", max_depth=3, learning_rate=0.1),
    dict(model="gbm", max_depth=4, learning_rate=0.1),
]
for s in base_grid:
    name = s.pop("run_name", None) or "-".join(f"{k}={v}" for k, v in s.items())
    train_and_log(**s, run_name=name)
print(len(mlflow.search_runs(experiment_names=[EXPERIMENT])), "runs logged")

5 runs logged


## Exercise 1

In `1_tracking.ipynb`, add a run with `model="logistic", C=0.01`. Where does it rank? Find it in the UI with the search box: `params.C = "0.01"`.

In [3]:
# The new run: exactly the same call as the grid, with C=0.01
train_and_log(model="logistic", C=0.01, run_name="model=logistic-C=0.01")

cols = ["tags.mlflow.runName", "metrics.roc_auc", "metrics.sensitivity"]
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["metrics.roc_auc DESC"])
runs["rank_by_auc"] = range(1, len(runs) + 1)
runs["rank_by_sensitivity"] = runs["metrics.sensitivity"].rank(ascending=False, method="min").astype(int)
runs[cols + ["rank_by_auc", "rank_by_sensitivity"]].round(4)

,tags.mlflow.runName,metrics.roc_auc,metrics.sensitivity,rank_by_auc,rank_by_sensitivity
0,baseline-logistic,0.8122,0.5702,1,5
1,model=logistic-C=0.1,0.8119,0.5730,2,4
2,model=logistic-C=0.01,0.8104,0.5785,3,1
3,model=gbm-max_depth=2-learning_rate=0.05,0.8071,0.5758,4,2
4,model=gbm-max_depth=3-learning_rate=0.1,0.8038,0.5675,5,6
5,model=gbm-max_depth=4-learning_rate=0.1,0.8028,0.5758,6,2


In [4]:
# The same thing the UI search box does (type  params.C = "0.01"  there)
mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string="params.C = '0.01'")[cols].round(4)

,tags.mlflow.runName,metrics.roc_auc,metrics.sensitivity
0,model=logistic-C=0.01,0.8104,0.5785


**Answer.** The `C=0.01` run ranks **3rd of 6 by ROC-AUC (0.8104)**: just behind the two other logistic runs
(0.8122 and 0.8119) and ahead of all three gbm runs (0.8071 down to 0.8028). Strong regularisation costs almost nothing here.
By **sensitivity** it ranks **1st (0.5785)**, so "best" depends on the metric you sort by. The search `params.C = "0.01"`
returns only this run (MLflow stores params as strings, hence the quotes); the gbm runs have no `C` param, so they never match.

## Exercise 2

Log one more metric per run: the share of flagged children who are under 2 years old. Where would you add it, and what would a clinic learn from it?

In [5]:
# Where: in train_and_log, right after em.evaluate, next to the other mlflow.log_metrics calls.
# To avoid editing train.py here, we re-open each finished run and add the metric to it.
def share_flagged_under_2(model, df, alert_rate=0.2):
    score = model.predict_proba(df[em.FEATURES])[:, 1]
    flag = score >= np.quantile(score, 1 - alert_rate)           # same top-20% rule as em.evaluate
    return float((df.loc[flag, "age_months"] < 24).mean())

test = em.load_months("2024-11", "2024-12")
print(f"Under-2s are {(test.age_months < 24).mean():.1%} of visits and "
      f"{(test.loc[test[em.LABEL] == 1, 'age_months'] < 24).mean():.1%} of the children who develop GAM.")

for run in mlflow.search_runs(experiment_names=[EXPERIMENT], output_format="list"):
    model_uri = mlflow.search_logged_models(filter_string=f"source_run_id = '{run.info.run_id}'",
                                            experiment_ids=[exp_id], output_format="list")[0].model_uri
    m_ = mlflow.sklearn.load_model(model_uri)
    with mlflow.start_run(run_id=run.info.run_id):               # re-open the finished run
        mlflow.log_metric("flagged_under_2_share", share_flagged_under_2(m_, test))

runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["metrics.flagged_under_2_share DESC"])
runs[["tags.mlflow.runName", "metrics.flagged_under_2_share", "metrics.roc_auc"]].round(4)

Under-2s are 68.5% of visits and 76.9% of the children who develop GAM.


,tags.mlflow.runName,metrics.flagged_under_2_share,metrics.roc_auc
0,model=logistic-C=0.1,0.7925,0.8119
1,model=gbm-max_depth=4-learning_rate=0.1,0.7900,0.8028
2,model=logistic-C=0.01,0.7887,0.8104
3,model=gbm-max_depth=2-learning_rate=0.05,0.7875,0.8071
4,baseline-logistic,0.7862,0.8122
5,model=gbm-max_depth=3-learning_rate=0.1,0.7762,0.8038


**Answer.** Add it in `train_and_log`, directly after `em.evaluate(...)`, as one more entry in the logged metrics
(`flagged_under_2_share`; the cells above do the same from outside by re-opening each finished run).
In the test months the under-2s are 68.5% of visits and 76.9% of the children who develop GAM, and
**every model flags 78% to 79% under-2s** (0.776 to 0.793, table above), so the follow-up capacity goes mostly to the youngest, slightly
more than their share of cases. A clinic learns who its 20% of follow-ups actually reaches, and whether that matches where
the cases are. The models barely differ on it, so it would not choose between them, but it would show if a retrained
model suddenly stopped flagging infants, or if older children's cases are being missed. It describes who is flagged,
not who is right, so read it alongside sensitivity by age band.

## Exercise 3

Change the promotion rule to also require that sensitivity in **Addis Ababa** doesn't drop. Does the challenger still pass? (`metrics_by_group.json` has the numbers.)

In [6]:
# Notebook 2: register the best 2024 run as version 1 (@champion), train the challenger, compare on May-Jun 2025.
MODEL = "gam-risk"
def model_uri_for(run_id):
    return mlflow.search_logged_models(filter_string=f"source_run_id = '{run_id}'", experiment_ids=[exp_id],
                                       output_format="list")[0].model_uri

cand = mlflow.search_runs(experiment_names=[EXPERIMENT],
                          filter_string="params.test_until = '2024-12' and metrics.roc_auc > 0",
                          order_by=["metrics.roc_auc DESC", "attributes.start_time ASC"])
best = cand.iloc[0]
v1 = mlflow.register_model(model_uri_for(best.run_id), MODEL)
client.set_registered_model_alias(MODEL, "champion", v1.version)
print("v1 =", best["tags.mlflow.runName"])

challenger_run, _, challenger_uri = train_and_log(model="logistic", C=1.0, train_until="2025-04",
                                                  test_from="2025-05", test_until="2025-06",
                                                  run_name="challenger-retrained-2025-04")
benchmark = em.load_months("2025-05", "2025-06")
champion_sk = mlflow.sklearn.load_model(f"models:/{MODEL}@champion")
challenger = mlflow.sklearn.load_model(challenger_uri)

rows = {}
for name, mod in [("champion (v1)", champion_sk), ("challenger", challenger)]:
    overall, by_group = em.evaluate(mod, benchmark)
    rows[name] = {"roc_auc": overall["roc_auc"], "sensitivity": overall["sensitivity"], "ppv": overall["ppv"],
                  "sens_girls": by_group["sex"]["F"]["sensitivity"], "sens_boys": by_group["sex"]["M"]["sensitivity"],
                  "sens_addis": by_group["region"]["Addis Ababa"]["sensitivity"],
                  "addis_cases": round(by_group["region"]["Addis Ababa"]["n"] * by_group["region"]["Addis Ababa"]["prevalence"])}
comparison = pd.DataFrame(rows).T
comparison.round(4)

Successfully registered model 'gam-risk'.
Created version '1' of model 'gam-risk'.


v1 = baseline-logistic


,roc_auc,sensitivity,ppv,sens_girls,sens_boys,sens_addis,addis_cases
champion (v1),0.8256,0.4897,0.5950,0.4719,0.5071,0.2143,28.0
challenger,0.8363,0.5206,0.6325,0.4969,0.5438,0.1429,28.0


In [7]:
def should_promote(champ, chall, min_auc_gain=0.005, max_group_drop=0.02, check_addis=False):
    reasons = []
    if chall["roc_auc"] - champ["roc_auc"] < min_auc_gain:
        reasons.append(f"ROC-AUC gain {chall['roc_auc'] - champ['roc_auc']:+.4f} < {min_auc_gain}")
    for g in ["sens_girls", "sens_boys"]:
        if champ[g] - chall[g] > max_group_drop:
            reasons.append(f"{g} drops by {champ[g] - chall[g]:.3f}")
    if check_addis and chall["sens_addis"] < champ["sens_addis"]:     # the new condition: no drop at all
        reasons.append(f"Addis Ababa sensitivity drops by {champ['sens_addis'] - chall['sens_addis']:.3f}")
    return not reasons, reasons

champ_row, chall_row = comparison.loc["champion (v1)"], comparison.loc["challenger"]
print("original rule:      ", should_promote(champ_row, chall_row))
print("with Addis check:   ", should_promote(champ_row, chall_row, check_addis=True))

# The same numbers are in the run's artifact, as the exercise says:
import json
path = mlflow.artifacts.download_artifacts(run_id=challenger_run, artifact_path="metrics_by_group.json")
addis = json.load(open(path))["region"]["Addis Ababa"]
print("from metrics_by_group.json (challenger):", {k: round(v, 4) for k, v in addis.items()})

original rule:       (True, [])
with Addis check:    (False, ['Addis Ababa sensitivity drops by 0.071'])


from metrics_by_group.json (challenger): {'n': 342, 'prevalence': 0.0819, 'roc_auc': 0.8146, 'sensitivity': 0.1429, 'ppv': 0.5714, 'alert_rate': 0.0205}


**Answer.** **No: the challenger now fails.** Addis Ababa sensitivity on the May-Jun 2025 benchmark falls from
**0.2143 (champion) to 0.1429 (challenger)**, a drop of 0.071, so the new condition refuses a challenger that the original rule promotes
(better overall: ROC-AUC 0.8256 to 0.8363, sensitivity 0.4897 to 0.5206, girls and boys both up).
Caution: Addis Ababa has only 28 cases in these two months, so 0.071 is **two children** (6 caught versus 4). A rule of "must not
drop at all" on a group this small can reject a good model by chance. A tolerance (like the 0.02 for girls and boys), a longer benchmark,
or a bootstrap interval would be safer. The same fact is in the challenger run's `metrics_by_group.json` (`sensitivity` 0.1429, `n` 342).

## Exercise 4

Roll back: point `@champion` at version 1, reload `models:/gam-risk@champion`, and confirm the January 2025 risk scores change back.

In [8]:
# First do what notebook 2 does: promote the challenger as version 2, and score January 2025 with it.
v2 = mlflow.register_model(challenger_uri, MODEL)
client.set_registered_model_alias(MODEL, "previous_champion", v1.version)
client.set_registered_model_alias(MODEL, "champion", v2.version)

january = em.load_months("2025-01", "2025-01")
def champion_risk():
    mv = client.get_model_version_by_alias(MODEL, "champion").version
    model = mlflow.pyfunc.load_model(f"models:/{MODEL}@champion")       # reload by alias, as the EMR would
    return mv, model.predict(january[em.FEATURES])[:, 1]

mv_before, risk_v2 = champion_risk()
print("champion is version", mv_before)

# The rollback itself: one line
client.set_registered_model_alias(MODEL, "champion", "1")

mv_after, risk_v1 = champion_risk()
print("champion is version", mv_after)

# Confirm: v1 scores differ from v2's, and equal the scores of the original v1 model
original_v1 = mlflow.pyfunc.load_model(f"models:/{MODEL}/1").predict(january[em.FEATURES])[:, 1]
print(f"mean |v2 - v1| risk: {np.abs(risk_v2 - risk_v1).mean():.4f}   mean risk v2 {risk_v2.mean():.3f} -> v1 {risk_v1.mean():.3f}")
print("rolled-back scores identical to version 1:", np.allclose(risk_v1, original_v1))
print("visits whose top-20% flag changes:",
      int(((risk_v1 >= np.quantile(risk_v1, .8)) != (risk_v2 >= np.quantile(risk_v2, .8))).sum()), "of", len(january))

champion is version 2


Registered model 'gam-risk' already exists. Creating a new version of this model...
Created version '2' of model 'gam-risk'.


champion is version 1


mean |v2 - v1| risk: 0.0339   mean risk v2 0.125 -> v1 0.092
rolled-back scores identical to version 1: True
visits whose top-20% flag changes: 54 of 2000


**Answer.** After `set_registered_model_alias(MODEL, "champion", "1")` and reloading `models:/gam-risk@champion`,
the alias resolves to version 1 again (printed above), and the January 2025 scores are exactly version 1's scores (`True`) and
differ from version 2's: the mean risk per child goes from 0.125 under v2 back to 0.092 under v1 (mean absolute change 0.034), and 54 of the
2,000 visits change between flagged and not flagged. The model has to be **reloaded**: a process that loaded the champion earlier keeps
the old object until it reloads.

## Exercise 5

Use `train.py` from the terminal to log a run, then find it in the UI. What does the UI show under *Source* compared with the notebook runs?

In [9]:
# Run train.py exactly as in a terminal, against the temporary store (env var set in the first cell)
out = subprocess.run([sys.executable, "train.py", "--model", "logistic", "--C", "0.1", "--run-name", "from-terminal"],
                     capture_output=True, text=True, env=os.environ)
print(out.stdout.strip() or out.stderr[-500:])

def source_info(name):
    r = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string=f"tags.mlflow.runName = '{name}'",
                           output_format="list")[0]
    t = r.data.tags
    return {k: t.get(k) for k in ["mlflow.source.type", "mlflow.source.name", "mlflow.source.git.commit",
                                  "mlflow.user", "git_commit"]}

pd.DataFrame({"notebook run (baseline-logistic)": source_info("baseline-logistic"),
              "terminal run (from-terminal)": source_info("from-terminal")})

run 8128538c7cde471788f851f28879af08: roc_auc=0.8119 sensitivity=0.5730  model models:/m-4ecec22a814849e3b8c2381d50ebadf1


,notebook run (baseline-logistic),terminal run (from-terminal)
mlflow.source.type,NOTEBOOK,LOCAL
mlflow.source.name,/home/user/mlops-workshop/.venv/lib/python3.12/site-packages/ipykernel_launcher.py,train.py
mlflow.source.git.commit,NaN,7c47a1924cbd357e232e4106ac59cebd1c710e2d
mlflow.user,root,root
git_commit,7c47a19,7c47a19


**Answer.** The *Source* column of the UI comes from the run tags `mlflow.source.name` and `mlflow.source.type` (table above).
The terminal run shows the **script**: source `train.py`, type `LOCAL`, plus `mlflow.source.git.commit` (the commit the code was at).
The notebook runs show type `NOTEBOOK` and, as source, the path of the **Jupyter kernel launcher** inside the virtualenv
(`.../ipykernel_launcher.py`), not the notebook's file name, and no git commit tag. So from *Source* alone you cannot tell which notebook
made a run, or which code version; our own `git_commit` tag (identical in both) is what carries that. A run from `train.py` is also
reproducible with one command, which is why a pipeline would call the script.
I could not show the UI itself (no browser here); the table shows the tags that column is built from.

In [10]:
shutil.rmtree(TMP, ignore_errors=True)       # remove the temporary store
print("temporary store removed")

temporary store removed
